# 🌧️ Seattle Weather Prediction using Deep Learning
## ANN, RNN, LSTM, GRU - Classification Task
**Dataset:** Seattle Weather Prediction Dataset

This notebook classifies weather types (drizzle, fog, rain, snow, sun) using
ANN, RNN, LSTM, and GRU architectures built with TensorFlow/Keras.

---
## 1. 📦 Import Libraries

In [ ]:

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
import os
import pickle
import json

warnings.filterwarnings('ignore')
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

import tensorflow as tf
from tensorflow.keras.models import Sequential, load_model
from tensorflow.keras.layers import Dense, Dropout, SimpleRNN, LSTM, GRU, Reshape
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.utils import to_categorical

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             confusion_matrix, classification_report, roc_auc_score, roc_curve)

print(f"TensorFlow version: {tf.__version__}")
print(f"NumPy version: {np.__version__}")
print(f"Pandas version: {pd.__version__}")


---
## 2. 📊 Dataset Understanding

### 2.1 Load Dataset

In [ ]:
df = pd.read_csv('Seattle Weather Prediction Dataset/seattle-weather.csv')

print("=" * 60)
print("DATASET OVERVIEW")
print("=" * 60)
print(f"\nShape: {df.shape}")
print(f"Rows: {df.shape[0]}, Columns: {df.shape[1]}")

print("\n--- First 5 Rows ---")
print(df.head())

print("\n--- Data Types ---")
print(df.dtypes)

print("\n--- Basic Statistics ---")
print(df.describe())

print("\n--- Missing Values ---")
print(df.isnull().sum())
print(f"\nTotal missing: {df.isnull().sum().sum()}")

# Identify numeric vs categorical
numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
cat_cols = df.select_dtypes(exclude=[np.number]).columns.tolist()
print(f"\nNumeric features ({len(numeric_cols)}): {numeric_cols}")
print(f"Categorical features ({len(cat_cols)}): {cat_cols}")


### 2.2 Target Variable Analysis

In [ ]:
print("\n--- Target Variable: 'weather' ---")
print(df['weather'].value_counts())


### 2.3 Visualizations

In [ ]:

# Missing value heatmap
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.heatmap(df.isnull(), cbar=True, yticklabels=False, ax=axes[0], cmap='viridis')
axes[0].set_title('Missing Value Heatmap', fontsize=14, fontweight='bold')

# Target class distribution
df['weather'].value_counts().plot(kind='bar', ax=axes[1], color=sns.color_palette("husl", 5), edgecolor='black')
axes[1].set_title('Weather Class Distribution', fontsize=14, fontweight='bold')
axes[1].set_xlabel('Weather Type')
axes[1].set_ylabel('Count')
plt.tight_layout()
plt.savefig('seattle_class_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

# Histograms of numeric features
df[numeric_cols].hist(bins=30, figsize=(14, 8), edgecolor='black', color='#4ECDC4')
plt.suptitle('Feature Distributions', fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('seattle_histograms.png', dpi=150, bbox_inches='tight')
plt.show()

# Correlation heatmap
plt.figure(figsize=(8, 6))
corr = df[numeric_cols].corr()
sns.heatmap(corr, annot=True, cmap='coolwarm', center=0, fmt='.2f',
            linewidths=0.5, square=True)
plt.title('Feature Correlation Heatmap', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('seattle_correlation.png', dpi=150, bbox_inches='tight')
plt.show()

# Time series plots
df['date'] = pd.to_datetime(df['date'])
fig, axes = plt.subplots(2, 2, figsize=(16, 10))
for i, col in enumerate(numeric_cols):
    ax = axes[i // 2, i % 2]
    ax.plot(df['date'], df[col], alpha=0.7, linewidth=0.5)
    ax.set_title(f'{col} Over Time', fontsize=12, fontweight='bold')
    ax.set_xlabel('Date')
    ax.set_ylabel(col)
plt.suptitle('Time Series Plots of Features', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.savefig('seattle_timeseries.png', dpi=150, bbox_inches='tight')
plt.show()


---
## 3. 🔧 Preprocessing

**Why each step:**
- **Drop date column:** Neural networks need numeric input, date is used for indexing only.
- **Label encoding:** Convert categorical target to numeric labels.
- **Feature scaling:** StandardScaler normalizes features to zero mean, unit variance - 
  critical for neural network convergence.
- **Reshape for RNN/LSTM/GRU:** These architectures expect 3D input (samples, timesteps, features).

In [ ]:

# Drop date column
X = df.drop(['date', 'weather'], axis=1).values
y = df['weather'].values

# Label encode target
le = LabelEncoder()
y_encoded = le.fit_transform(y)
print(f"Classes: {le.classes_}")
print(f"Encoded labels: {np.unique(y_encoded)}")
n_classes = len(le.classes_)

# Scale features
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y_encoded, test_size=0.2, random_state=42, stratify=y_encoded
)
print(f"\nTrain shape: {X_train.shape}, Test shape: {X_test.shape}")

# One-hot encode for classification
y_train_cat = to_categorical(y_train, n_classes)
y_test_cat = to_categorical(y_test, n_classes)

# Reshape for RNN models: (samples, timesteps=1, features)
X_train_rnn = X_train.reshape(X_train.shape[0], 1, X_train.shape[1])
X_test_rnn = X_test.reshape(X_test.shape[0], 1, X_test.shape[1])
print(f"RNN input shape: {X_train_rnn.shape}")


---
## 4. 🏗️ Model Building

We build 4 models: **ANN**, **Simple RNN**, **LSTM**, **GRU**

**Key components in each:**
- **Dropout (0.3):** Regularization to prevent overfitting by randomly dropping neurons
- **EarlyStopping:** Stops training when validation loss stops improving
- **Adam optimizer:** Adaptive learning rate optimizer, excellent default choice
- **Categorical crossentropy:** Standard loss for multi-class classification
- **Softmax activation:** Outputs probability distribution over classes

In [ ]:

early_stop = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True, verbose=1)
EPOCHS = 100
BATCH = 32

results = {}


### 4.1 ANN (Artificial Neural Network)

In [ ]:
print("\n" + "=" * 60)
print("MODEL 1: ANN (Artificial Neural Network)")
print("=" * 60)

model_ann = Sequential([
    Dense(128, activation='relu', input_shape=(X_train.shape[1],)),
    Dropout(0.3),
    Dense(64, activation='relu'),
    Dropout(0.3),
    Dense(32, activation='relu'),
    Dense(n_classes, activation='softmax')
])
model_ann.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
model_ann.summary()

history_ann = model_ann.fit(X_train, y_train_cat, epochs=EPOCHS, batch_size=BATCH,
                            validation_split=0.2, callbacks=[early_stop], verbose=1)


### 4.2 Simple RNN

In [ ]:
print("\n" + "=" * 60)
print("MODEL 2: Simple RNN")
print("=" * 60)

model_rnn = Sequential([
    SimpleRNN(128, activation='tanh', input_shape=(1, X_train.shape[1]), return_sequences=True),
    Dropout(0.3),
    SimpleRNN(64, activation='tanh'),
    Dropout(0.3),
    Dense(32, activation='relu'),
    Dense(n_classes, activation='softmax')
])
model_rnn.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
model_rnn.summary()

history_rnn = model_rnn.fit(X_train_rnn, y_train_cat, epochs=EPOCHS, batch_size=BATCH,
                            validation_split=0.2, callbacks=[early_stop], verbose=1)


### 4.3 LSTM

In [ ]:
print("\n" + "=" * 60)
print("MODEL 3: LSTM (Long Short-Term Memory)")
print("=" * 60)

model_lstm = Sequential([
    LSTM(128, activation='tanh', input_shape=(1, X_train.shape[1]), return_sequences=True),
    Dropout(0.3),
    LSTM(64, activation='tanh'),
    Dropout(0.3),
    Dense(32, activation='relu'),
    Dense(n_classes, activation='softmax')
])
model_lstm.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
model_lstm.summary()

history_lstm = model_lstm.fit(X_train_rnn, y_train_cat, epochs=EPOCHS, batch_size=BATCH,
                              validation_split=0.2, callbacks=[early_stop], verbose=1)


### 4.4 GRU

In [ ]:
print("\n" + "=" * 60)
print("MODEL 4: GRU (Gated Recurrent Unit)")
print("=" * 60)

model_gru = Sequential([
    GRU(128, activation='tanh', input_shape=(1, X_train.shape[1]), return_sequences=True),
    Dropout(0.3),
    GRU(64, activation='tanh'),
    Dropout(0.3),
    Dense(32, activation='relu'),
    Dense(n_classes, activation='softmax')
])
model_gru.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
model_gru.summary()

history_gru = model_gru.fit(X_train_rnn, y_train_cat, epochs=EPOCHS, batch_size=BATCH,
                            validation_split=0.2, callbacks=[early_stop], verbose=1)


---
## 5. 📈 Training Visualization & Overfitting Analysis

**Overfitting** occurs when a model learns training data too well (including noise)
but fails to generalize to unseen data. Signs: training accuracy >> validation accuracy.

**How we combat overfitting:**
- **Dropout:** Randomly deactivates neurons during training, forcing redundancy
- **EarlyStopping:** Halts training before overfitting worsens

In [ ]:

models_info = {
    'ANN': history_ann,
    'RNN': history_rnn,
    'LSTM': history_lstm,
    'GRU': history_gru
}

fig, axes = plt.subplots(2, 4, figsize=(24, 10))
for i, (name, hist) in enumerate(models_info.items()):
    # Accuracy curves
    axes[0, i].plot(hist.history['accuracy'], label='Train', linewidth=2)
    axes[0, i].plot(hist.history['val_accuracy'], label='Validation', linewidth=2)
    axes[0, i].set_title(f'{name} - Accuracy', fontsize=13, fontweight='bold')
    axes[0, i].set_xlabel('Epoch')
    axes[0, i].set_ylabel('Accuracy')
    axes[0, i].legend()
    axes[0, i].grid(True, alpha=0.3)
    
    # Loss curves
    axes[1, i].plot(hist.history['loss'], label='Train', linewidth=2)
    axes[1, i].plot(hist.history['val_loss'], label='Validation', linewidth=2)
    axes[1, i].set_title(f'{name} - Loss', fontsize=13, fontweight='bold')
    axes[1, i].set_xlabel('Epoch')
    axes[1, i].set_ylabel('Loss')
    axes[1, i].legend()
    axes[1, i].grid(True, alpha=0.3)

plt.suptitle('Training vs Validation Curves', fontsize=18, fontweight='bold')
plt.tight_layout()
plt.savefig('seattle_training_curves.png', dpi=150, bbox_inches='tight')
plt.show()


---
## 6. 📋 Evaluation

In [ ]:

def evaluate_model(model, X_te, y_te, y_te_cat, model_name, is_rnn=False):
    """Evaluate a model and return metrics dict."""
    y_pred_prob = model.predict(X_te)
    y_pred = np.argmax(y_pred_prob, axis=1)
    y_true = np.argmax(y_te_cat, axis=1) if len(y_te_cat.shape) > 1 else y_te
    
    acc = accuracy_score(y_true, y_pred)
    prec = precision_score(y_true, y_pred, average='weighted', zero_division=0)
    rec = recall_score(y_true, y_pred, average='weighted', zero_division=0)
    f1 = f1_score(y_true, y_pred, average='weighted', zero_division=0)
    
    try:
        auc = roc_auc_score(y_te_cat, y_pred_prob, multi_class='ovr', average='weighted')
    except:
        auc = 0.0
    
    print(f"\n{'=' * 50}")
    print(f"  {model_name} Evaluation Results")
    print(f"{'=' * 50}")
    print(f"  Accuracy:  {acc:.4f}")
    print(f"  Precision: {prec:.4f}")
    print(f"  Recall:    {rec:.4f}")
    print(f"  F1-Score:  {f1:.4f}")
    print(f"  ROC-AUC:   {auc:.4f}")
    print(f"\nClassification Report:\n{classification_report(y_true, y_pred, target_names=le.classes_)}")
    
    return {'Model': model_name, 'Accuracy': acc, 'Precision': prec,
            'Recall': rec, 'F1-Score': f1, 'ROC-AUC': auc,
            'y_pred': y_pred, 'y_pred_prob': y_pred_prob, 'y_true': y_true}

r_ann = evaluate_model(model_ann, X_test, y_test, y_test_cat, 'ANN')
r_rnn = evaluate_model(model_rnn, X_test_rnn, y_test, y_test_cat, 'RNN', True)
r_lstm = evaluate_model(model_lstm, X_test_rnn, y_test, y_test_cat, 'LSTM', True)
r_gru = evaluate_model(model_gru, X_test_rnn, y_test, y_test_cat, 'GRU', True)

all_results = [r_ann, r_rnn, r_lstm, r_gru]


### Confusion Matrices

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(24, 5))
for i, r in enumerate(all_results):
    cm = confusion_matrix(r['y_true'], r['y_pred'])
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[i],
                xticklabels=le.classes_, yticklabels=le.classes_)
    axes[i].set_title(f"{r['Model']} Confusion Matrix", fontsize=12, fontweight='bold')
    axes[i].set_xlabel('Predicted')
    axes[i].set_ylabel('Actual')
plt.tight_layout()
plt.savefig('seattle_confusion_matrices.png', dpi=150, bbox_inches='tight')
plt.show()


### ROC Curves

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(24, 5))
for i, r in enumerate(all_results):
    for c in range(n_classes):
        fpr, tpr, _ = roc_curve(y_test_cat[:, c], r['y_pred_prob'][:, c])
        axes[i].plot(fpr, tpr, label=f'{le.classes_[c]}', linewidth=2)
    axes[i].plot([0, 1], [0, 1], 'k--', alpha=0.5)
    axes[i].set_title(f"{r['Model']} ROC Curves", fontsize=12, fontweight='bold')
    axes[i].set_xlabel('False Positive Rate')
    axes[i].set_ylabel('True Positive Rate')
    axes[i].legend(fontsize=8)
plt.tight_layout()
plt.savefig('seattle_roc_curves.png', dpi=150, bbox_inches='tight')
plt.show()


---
## 7. 📊 Comparative Analysis

In [ ]:

comparison_df = pd.DataFrame([{k: v for k, v in r.items() 
                               if k not in ['y_pred', 'y_pred_prob', 'y_true']} 
                              for r in all_results])
comparison_df = comparison_df.set_index('Model')
print("\n" + "=" * 70)
print("  MODEL COMPARISON TABLE")
print("=" * 70)
print(comparison_df.round(4).to_string())

# Comparison bar chart
comparison_df[['Accuracy', 'Precision', 'Recall', 'F1-Score', 'ROC-AUC']].plot(
    kind='bar', figsize=(12, 6), edgecolor='black', width=0.8)
plt.title('Model Performance Comparison', fontsize=16, fontweight='bold')
plt.ylabel('Score')
plt.ylim(0, 1.1)
plt.xticks(rotation=0)
plt.legend(loc='lower right')
plt.tight_layout()
plt.savefig('seattle_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

best_model_name = comparison_df['Accuracy'].idxmax()
print(f"\n🏆 Best performing model: {best_model_name} with accuracy {comparison_df.loc[best_model_name, 'Accuracy']:.4f}")


### Comparative Discussion

In [ ]:
print("""
╔══════════════════════════════════════════════════════════════════╗
║                    COMPARATIVE ANALYSIS                        ║
╠══════════════════════════════════════════════════════════════════╣
║                                                                ║
║ 1. VANISHING GRADIENT PROBLEM:                                 ║
║    - Simple RNNs suffer from vanishing gradients during        ║
║      backpropagation through time (BPTT), making it hard       ║
║      to learn long-range dependencies.                         ║
║    - LSTM uses forget/input/output gates to control info flow. ║
║    - GRU uses reset/update gates (simpler than LSTM).          ║
║                                                                ║
║ 2. WHY LSTM/GRU MAY OUTPERFORM RNN:                           ║
║    - Gate mechanisms allow selective memory retention.          ║
║    - Better gradient flow during training.                     ║
║    - Can capture both short and long-term patterns.            ║
║                                                                ║
║ 3. COMPUTATIONAL COMPLEXITY:                                   ║
║    - ANN: Fastest, no recurrence overhead.                     ║
║    - RNN: Moderate, simple recurrent connections.              ║
║    - LSTM: Most complex (3 gates per cell).                    ║
║    - GRU: Between RNN and LSTM (2 gates per cell).             ║
║                                                                ║
║ 4. DATASET-SPECIFIC OBSERVATIONS:                              ║
║    - Seattle weather has 5 classes, imbalanced (rain dominant).║
║    - With tabular (non-sequential) input, ANN can compete.    ║
║    - Sequence models benefit more from time-windowed data.     ║
║                                                                ║
╚══════════════════════════════════════════════════════════════════╝
""")


---
## 8. 💾 Model Saving

**Why model persistence is important:**
- Avoids retraining (saves time and compute resources)
- Enables deployment to production environments
- Allows sharing models with team members
- Required for the Streamlit dashboard

In [ ]:

os.makedirs('saved_models', exist_ok=True)

model_ann.save('saved_models/seattle_ann_model.keras')
model_rnn.save('saved_models/seattle_rnn_model.keras')
model_lstm.save('saved_models/seattle_lstm_model.keras')
model_gru.save('saved_models/seattle_gru_model.keras')

with open('saved_models/seattle_scaler.pkl', 'wb') as f:
    pickle.dump(scaler, f)
with open('saved_models/seattle_label_encoder.pkl', 'wb') as f:
    pickle.dump(le, f)

# Save comparison results
comparison_df.to_csv('saved_models/seattle_comparison_results.csv')

print("\n✅ All models and preprocessing objects saved successfully!")
print("   📁 saved_models/seattle_ann_model.keras")
print("   📁 saved_models/seattle_rnn_model.keras")
print("   📁 saved_models/seattle_lstm_model.keras")
print("   📁 saved_models/seattle_gru_model.keras")
print("   📁 saved_models/seattle_scaler.pkl")
print("   📁 saved_models/seattle_label_encoder.pkl")
print("\n🎉 Seattle Weather Prediction Notebook Complete!")
